# 32 — Uncertainty, Calibration, Ensembling, and Selective Prediction

A neural network can be highly confident and still be wrong.

For medical imaging, we often need to understand:

- Prediction confidence
- Calibration
- Model uncertainty
- Domain-shift uncertainty
- When a model should abstain

We will study:

- Reliability diagrams
- Expected calibration error
- Temperature scaling
- Monte Carlo dropout
- Deep ensembles
- Predictive entropy
- Uncertainty under domain shift
- Selective prediction
- Risk–coverage curves


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
print("PyTorch:",torch.__version__)


# 1. Confidence

For multiclass prediction:

$$
confidence=\max_c p(y=c|x)
$$

High confidence is not a guarantee of correctness.


In [ ]:
p=torch.tensor([[0.95,0.03,0.02],[0.40,0.35,0.25]])
print(p.max(dim=1).values)


# 2. Calibration

A calibrated model should satisfy:

> Among predictions made with about 80% confidence, roughly 80% should be correct.


# 3. Reliability Bins


In [ ]:
def calibration_bins(probabilities,targets,n_bins=10):
    conf,pred=probabilities.max(dim=1)
    correct=(pred==targets).float()
    edges=torch.linspace(0,1,n_bins+1)
    rows=[]
    for i in range(n_bins):
        lo,hi=edges[i],edges[i+1]
        mask=(conf>=lo)&(conf<=hi if i==n_bins-1 else conf<hi)
        if mask.any():
            rows.append({
                "count":int(mask.sum()),
                "confidence":float(conf[mask].mean()),
                "accuracy":float(correct[mask].mean())
            })
    return rows


# 4. Expected Calibration Error

$$
\boxed{
ECE=\sum_b\frac{|B_b|}{N}|acc(B_b)-conf(B_b)|
}
$$


In [ ]:
def ece(probabilities,targets,n_bins=10):
    rows=calibration_bins(probabilities,targets,n_bins)
    n=len(targets)
    return sum((r["count"]/n)*abs(r["accuracy"]-r["confidence"]) for r in rows)


# 5. ECE Caveat

ECE depends on binning and sample size. Always document the implementation.


In [ ]:
targets=torch.tensor([0,0,1,2,1,0,2,2,1,0])
logits=torch.tensor([
    [5.,.2,.1],[4.,.1,0.],[.1,4.,.2],[4.,.2,.1],[.1,3.,.2],
    [3.,.2,.1],[.1,.2,3.],[0.,.5,2.5],[2.5,.4,.3],[2.,.8,.2]
])
probs=torch.softmax(logits,dim=1)
print("ECE:",ece(probs,targets,5))


# 6. Temperature Scaling

Scale logits:

$$
z'=\frac{z}{T}
$$

Usually $T>1$ softens overconfident predictions.


In [ ]:
def temperature_probs(logits,T):
    return torch.softmax(logits/T,dim=1)

for T in [0.5,1.0,2.0]:
    print(T,temperature_probs(logits[:1],T))


# 7. Fit Temperature on Validation Data Only


In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_T=nn.Parameter(torch.zeros(1))
    def forward(self,z):
        return z/self.log_T.exp()

def fit_temperature(logits,targets,steps=200,lr=0.05):
    s=TemperatureScaler()
    opt=torch.optim.Adam(s.parameters(),lr=lr)
    for _ in range(steps):
        loss=F.cross_entropy(s(logits),targets)
        opt.zero_grad(set_to_none=True)
        loss.backward();opt.step()
    return float(s.log_T.exp().item())


# 8. Temperature Scaling Does Not Usually Change Ranking

It adjusts probability sharpness. It generally does not improve discrimination such as AUROC.


# 9. Aleatoric vs Epistemic Uncertainty

Aleatoric:

- Inherent ambiguity/noise

Epistemic:

- Model/data uncertainty
- Limited training support


# 10. Monte Carlo Dropout

Keep dropout stochastic during repeated inference passes. Variation becomes an uncertainty signal.


In [ ]:
class DropoutNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(16,32),nn.ReLU(),nn.Dropout(0.4),nn.Linear(32,3)
        )
    def forward(self,x): return self.net(x)

def enable_mc_dropout(model):
    model.eval()
    for m in model.modules():
        if isinstance(m,nn.Dropout):
            m.train()

def mc_predict(model,x,samples=30):
    enable_mc_dropout(model)
    preds=[]
    with torch.no_grad():
        for _ in range(samples):
            preds.append(torch.softmax(model(x),dim=1))
    stack=torch.stack(preds)
    return stack.mean(0),stack.var(0)


# 11. BatchNorm Caveat

Do not simply call `model.train()` on a BatchNorm model for MC dropout; that also changes BatchNorm behavior. Activate dropout layers specifically.


# 12. Predictive Entropy

$$
\boxed{
H(p)=-\sum_cp_c\log p_c
}
$$


In [ ]:
def predictive_entropy(probabilities,eps=1e-8):
    return -(probabilities*(probabilities+eps).log()).sum(dim=1)

print(predictive_entropy(torch.tensor([[.98,.01,.01],[.34,.33,.33]])))


# 13. Deep Ensembles

Train multiple independent models:

$$
M_1,\ldots,M_K
$$

Average probabilities:

$$
p_{ens}=\frac{1}{K}\sum_kp_k
$$


In [ ]:
def ensemble_probabilities(probability_list):
    return torch.stack(probability_list).mean(dim=0)


# 14. Seed Ensembles

A practical ensemble baseline is to train the same architecture with several random seeds and average predictions.


# 15. Ensemble Diversity

Ensembling helps more when member errors are not identical. Diversity can come from different seeds, folds, or architectures.


# 16. Uncertainty Under Domain Shift

A model may become more uncertain—or dangerously remain confident—on a new scanner/site.

Evaluate uncertainty and calibration externally.


# 17. Softmax Is Not an OOD Detector

Out-of-distribution images can still receive very high softmax confidence.


# 18. Selective Prediction

A model can abstain when uncertainty is high.

Coverage:

$$
Coverage=\frac{Accepted}{Total}
$$


In [ ]:
def selective_accuracy(probabilities,targets,threshold):
    conf,pred=probabilities.max(dim=1)
    accepted=conf>=threshold
    if not accepted.any():
        return {"coverage":0.0,"accuracy":float("nan")}
    return {
        "coverage":float(accepted.float().mean()),
        "accuracy":float((pred[accepted]==targets[accepted]).float().mean())
    }


# 19. Risk–Coverage Curve

As coverage increases, the model accepts more difficult cases.

Risk is:

$$
1-Accuracy_{accepted}
$$


In [ ]:
def risk_coverage(probabilities,targets):
    conf,pred=probabilities.max(dim=1)
    order=torch.argsort(conf,descending=True)
    correct=(pred[order]==targets[order]).float()
    rows=[]
    for k in range(1,len(targets)+1):
        coverage=k/len(targets)
        risk=1-float(correct[:k].mean())
        rows.append((coverage,risk))
    return rows


# 20. Medical Abstention

Potential triggers:

- Poor image quality
- Unknown device
- Ensemble disagreement
- Ambiguous probability

Abstained cases can be routed for human review.


# 21. Selective Prediction Is Not a Safety Guarantee

A model may still be confidently wrong. Validate abstention behavior explicitly.


# 22. Calibration by Class and Site

Overall calibration may hide subgroup miscalibration. Check clinically relevant groups when sample size permits.


# 23. Common Mistakes

- Fitting temperature on test data
- Treating confidence as certainty
- Choosing ensemble members using test performance
- Ignoring calibration under domain shift
- Assuming abstention solves all failures


# 24. Exercises

1. Compute confidence.
2. Implement reliability bins.
3. Compute ECE.
4. Fit temperature.
5. Implement predictive entropy.
6. Run MC dropout.
7. Average an ensemble.
8. Implement abstention.
9. Plot risk–coverage.
10. Compare internal vs external calibration.


# 25. Key Takeaways

Calibration asks whether predicted probabilities match observed correctness.

Uncertainty methods provide useful signals but not guarantees.

Selective prediction introduces:

$$
\boxed{Predict\ or\ Abstain}
$$


# Next Notebook

# 33 — Efficient Training, Inference, and Model Deployment

In the next notebook, we will study DataLoader efficiency, `torch.compile`, profiling, latency, throughput, export, and reproducible deployment.
